# LoRA Dialect Fine-Tuner — Parameter-Efficient Fine-Tuning

**What you'll learn:** why fine-tuning a full LLM is expensive, what LoRA does differently, and how to measure parameter efficiency.

**Estimated time:** 45–60 min &nbsp;|&nbsp; **Runtime:** GPU

---

### The problem LoRA solves

Fine-tuning a large model means updating every one of its billions of parameters on your task. That requires a GPU with enough VRAM to hold both the model *and* the gradients — often impractical.

**LoRA (Low-Rank Adaptation)** observation: when we fine-tune, the change to each weight matrix `ΔW` tends to have low **intrinsic rank**. So instead of storing the full `ΔW`, we approximate it as two small matrices:

```
ΔW ≈ B x A     where A: (d x r) and B: (r x d)
```

If `d = 768` and `r = 8`: instead of `768 x 768 = 589,824` parameters, we store `768x8 + 8x768 = 12,288` — a **48x reduction**.

? **Before running:** If a model has 10 layers each with a 1024x1024 weight matrix, and we use LoRA with r=16, how many parameters are we training vs. the full model? Calculate it.


## 1  Install and load

In [ ]:
!pip -q install transformers peft datasets accelerate
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'distilgpt2'   # ~82M params — fast on free GPU
tok = AutoTokenizer.from_pretrained(model_name)
tok.pad_token = tok.eos_token   # GPT-2 has no pad token; use EOS instead

base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_params = sum(p.numel() for p in base_model.parameters())
print(f"Base model parameters: {base_params:,}")
print(f"That's ~{base_params/1e6:.0f}M parameters")

## 2  Understand LoRA parameters before configuring it

The `LoraConfig` has several key knobs:

- `r` — the rank. Higher = more capacity but more parameters. Start with 8.
- `lora_alpha` — a scaling factor. Convention: set it to 2x the rank.
- `target_modules` — which weight matrices to inject LoRA into. For GPT-2, `'c_attn'` targets the combined Q, K, V projection.
- `lora_dropout` — regularization; 0.05 is a safe default.

? **Predict:** With `r=8` and `target_modules=['c_attn']`, how many trainable parameters do you expect? (distilgpt2 has 6 attention layers, each `c_attn` is `768 x 2304`.)

Write your calculation here before running:
```
trainable ≈ 6 layers x 2 LoRA matrices x (768x8 + 8x2304) = ???
```


In [ ]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=8,                        # rank — the key hyperparameter
    lora_alpha=16,              # scaling: typically 2r
    target_modules=['c_attn'],  # which attention matrices to adapt
    lora_dropout=0.05,
    task_type='CAUSAL_LM',
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nThat's {100 * trainable / base_params:.4f}% of the base model.")

? **Was your calculation close?** What surprised you?

The tiny trainable percentage is why LoRA lets you fine-tune on a laptop GPU — you only store and compute gradients for those small matrices.


## 3  Build your training dataset

This is the most important step — the model will become what you teach it.

Below are 3 example sentences. **Your task:** add at least 5 more examples in the same style. Be consistent — use the same Q/A format and the same voice/dialect throughout.

Some ideas:
- Pirate speak (as below)
- Formal Victorian English
- Valley girl slang
- A specific character's voice from a show you know
- A technical domain (all answers as if explaining to a 5-year-old)

**Important:** The model will only learn the style you provide. Inconsistent examples confuse it.


In [ ]:
# YOUR TRAINING DATA — add at least 5 more lines in the same style
samples = [
    "Q: How are you? A: Arr, I be doin' grand, matey!",
    "Q: What time is it? A: The sun tells me it be high noon, ye landlubber!",
    "Q: What is the weather? A: The skies be clear, ye lucky sailor!",

    # ADD YOUR EXAMPLES HERE:
    # "Q: ... A: ...",
    "Q: Where are you from? A: I hail from the seven seas, with no port to call me own!",
    "Q: What do you want? A: Arr, I seek adventure and treasure beyond measure!",
    "Q: Do you have any food? A: Only hardtack and salt pork, but ye be welcome to share!",
    "Q: Are you tired? A: A true pirate never sleeps — only rests one eye, matey!",
    "Q: What is your job? A: I sail and plunder, as any respectable sea dog would do!",
]

print(f"Training samples: {len(samples)}")
print("\nEncoding...")
enc = tok(samples, return_tensors='pt', padding=True, truncation=True, max_length=64)
enc['labels'] = enc['input_ids'].clone()
print(f"Token tensor shape: {enc['input_ids'].shape}")

## 4  Generate before training — see the baseline

Before training, generate with the same prompts you'll test after. This is your **baseline** — write down what the model says so you can compare later.


In [ ]:
def generate(prompt, max_new=30):
    model.eval()
    ids = tok(prompt, return_tensors='pt')
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new, do_sample=True, temperature=0.9, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

test_prompts = ["Q: How are you? A:", "Q: What is the weather? A:", "Q: Where are you from? A:"]

print("=== BEFORE TRAINING ===")
for p in test_prompts:
    print(f"  {p}")
    print(f"  -> {generate(p)}")
    print()

## 5  Train the LoRA adapter

In [ ]:
model.train()
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)

losses = []
for step in range(80):
    out = model(**enc)
    loss = out.loss
    loss.backward()
    opt.step()
    opt.zero_grad()
    losses.append(loss.item())
    if step % 20 == 0:
        print(f"Step {step:3d}: loss={loss.item():.3f}")

import matplotlib.pyplot as plt
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('LoRA training loss')
plt.grid(alpha=0.3)
plt.show()

## 6  Generate after training — compare

In [ ]:
print("=== AFTER TRAINING ===")
for p in test_prompts:
    print(f"  {p}")
    print(f"  -> {generate(p)}")
    print()

? **How much did the style shift?** Compare to your before-training notes.

If the style didn't shift much, it's likely because:
1. Not enough training examples (add more)
2. Too few training steps (increase to 150+)
3. Training examples were inconsistent

If it's generating gibberish, loss might have spiked — try lowering `lr` to `5e-4`.


## 7  Save the adapter — note how small it is

In [ ]:
model.save_pretrained("./my_lora_adapter")
import os
size_bytes = sum(
    os.path.getsize(os.path.join("./my_lora_adapter", f))
    for f in os.listdir("./my_lora_adapter")
)
print(f"Adapter size on disk: {size_bytes / 1024:.1f} KB")
print(f"Full distilgpt2 size: ~330 MB")
print(f"Compression ratio: ~{330 * 1024 / (size_bytes / 1024):.0f}x")

The adapter is a few hundred KB. The base model is 330 MB. This is the practical power of LoRA — you ship a tiny adapter and the user already has (or downloads) the base model separately.


## 8  Challenges

1. **Better dataset:** Add 20+ examples in your chosen style. Does more data help more than more steps?

2. **Sweep the rank:** Try `r=4`, `r=16`, `r=32`. Plot the trainable parameter count vs. final loss. Where's the sweet spot?

3. **Load and merge:** Load the adapter back and merge it into the base model weights permanently:
```python
from peft import PeftModel
loaded = PeftModel.from_pretrained(base_model, "./my_lora_adapter")
merged = loaded.merge_and_unload()
```
What's the size of `merged`? Is it the same as the base model?

4. **Style transfer experiment:** Fine-tune on formal English Q&A, then test with casual prompts. Does the style transfer generalize to topics not in your training set?
